In [ ]:
import random
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

In [ ]:
# brew install graphviz
# pip install graphviz
from graphviz import Digraph
import torch
import torch.nn as nn

In [ ]:
np.random.seed(1337)
random.seed(1337)
torch.manual_seed(1337)

In [ ]:
# make up a dataset
from sklearn.datasets import make_moons, make_blobs
X, y = make_moons(n_samples=100, noise=0.2)
y = y*2 - 1  # make y be -1 or 1
# visualize in 2D
plt.figure(figsize=(5,5))
plt.scatter(X[:,0], X[:,1], c=y, s=20, cmap='jet')

In [ ]:
# initialize a model (MLP with 2 hidden layers)
model = nn.Sequential(
    nn.Linear(2, 16),
    nn.ReLU(),
    nn.Linear(16, 16),
    nn.ReLU(),
    nn.Linear(16, 1)
)
print(model)
# collect parameters list like micrograd's parameters()
params = list(model.parameters())
print("number of parameter tensors", len(params))

In [ ]:
X.shape[0]

In [ ]:
# loss function (SVM style max-margin)
def loss(batch_size=None):
    # inline DataLoader :)
    if batch_size is None:
        Xb, yb = X, y
    else:
        ri = np.random.permutation(X.shape[0])[:batch_size]
        Xb, yb = X[ri], y[ri]
    Xb_t = torch.tensor(Xb, dtype=torch.float32)
    yb_t = torch.tensor(yb, dtype=torch.float32)
    scores = model(Xb_t).squeeze(1)
    # svm max-margin loss: mean(relu(1 - y * score))
    losses = torch.relu(1 - yb_t * scores)
    data_loss = losses.mean()
    # L2 regularization
    alpha = 1e-4
    reg_loss = torch.stack([p.pow(2).sum() for p in model.parameters()]).sum() * alpha
    total_loss = data_loss + reg_loss
    # accuracy
    with torch.no_grad():
        preds = (scores > 0).to(torch.int8)
        truth = (yb_t > 0).to(torch.int8)
        accuracy = (preds == truth).float().mean().item()
    return total_loss, accuracy

total_loss, acc = loss()
print(total_loss.item(), acc)

In [ ]:
# optimization (manual SGD to mirror original)
for k in range(100):
    total_loss, acc = loss()
    model.zero_grad()
    total_loss.backward()
    learning_rate = 1.0 - 0.9 * k / 100
    for p in model.parameters():
        p.data -= learning_rate * p.grad
    if k % 1 == 0:
        print(f"step {k} loss {total_loss.item()}, accuracy {acc*100}%")

In [ ]:
# visualize decision boundary
h = 0.25
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Xmesh = np.c_[xx.ravel(), yy.ravel()]
Xmesh_t = torch.tensor(Xmesh, dtype=torch.float32)
scores = model(Xmesh_t).squeeze(1).detach().numpy()
Z = scores > 0
Z = Z.reshape(xx.shape)
fig = plt.figure()
plt.contourf(xx, yy, Z, cmap=plt.cm.Spectral, alpha=0.8)
plt.scatter(X[:, 0], X[:, 1], c=y, s=40, cmap=plt.cm.Spectral)
plt.xlim(xx.min(), xx.max())
plt.ylim(yy.min(), yy.max())